In [1]:
import torch
import numpy as np

from src.utils import (
    get_args,
    set_seed,
    get_datesets_and_loaders,
    get_trained_VAE,
    get_trained_VAE_with_domain_classifier,
    get_trained_classifier,
    get_trained_classifier_Base,
    test_model,
    prepare_report,
    run_all_senario
)
from src.tupl import run_tupl
from src.our_tupl import GENERATION_POLICIES, run_m1_tupl, run_all_senario_m1_tupl

/home/asad/workspace/DomainProject/changeDomain/notebooks/effective-gzsda/gzsda/src/utils.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy
/home/asad/workspace/anaconda3/envs/asad/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DOMAIN_SET =['Art','Clipart','Product','RealWorld']
DATA_DIR = './data/OfficeHome/'
DATASET_DETAILS = {
    "prefix": 'OfficeHome-',
    "suffix": '-resnet50-noft.mat',
    "resnet_feature": 'resnet50_features',
    "split_file_name": 'instanceSplit_officehome_unseen30.mat'
}
NUM_LABELS=65

In [3]:
import json
import time
from pathlib import Path

RESULT_OBJ_PATH = "./result/json/officeHome.json"
RESULT_CSV_PATH = "./result/csv/officeHome.csv"
RESULT_TIME_PATH = "./result/times/raw_time_officeHome.json"
RESULT_TIME_CSV_PATH = "./result/times/result_time_officeHome.csv"
path = Path(RESULT_OBJ_PATH)
time_path = Path(RESULT_TIME_PATH)

if path.exists():
    with path.open("r", encoding="utf-8") as f:
        result = json.load(f)
else:
    result = {}

if time_path.exists():
    with time_path.open("r", encoding="utf-8") as f:
        result_time = json.load(f)
else:
    result_time = {}

def save_results():
    time_path.parent.mkdir(parents=True, exist_ok=True)
    with open(RESULT_OBJ_PATH, "w") as f:
        json.dump(result, f, indent=2)
    with open(RESULT_TIME_PATH, "w") as f:
        json.dump(result_time, f, indent=2)

result.keys(), result_time.keys()


(dict_keys(['base', 'CCVAE', 'our0', 'our_GRE', 'TUPL', 'our_TUPL_real_plus_src2tgt', 'our_TUPL_real_plus_src2tgt_unseen', 'our_TUPL_interp_src2tgt']),
 dict_keys(['base', 'CCVAE', 'our0', 'our_GRE', 'TUPL', 'our_TUPL_real_plus_src2tgt', 'our_TUPL_real_plus_src2tgt_unseen', 'our_TUPL_interp_src2tgt']))

In [4]:
base = "base"
CCVAE = "CCVAE"
our0 = "our0"
our_GRE = "our_GRE"
tupl = "TUPL"
our_tupl = "our_TUPL"

# clear last result
# _, _ = result.pop(base, None), result_time.pop(base, None)
# _, _ = result.pop(CCVAE, None), result_time.pop(CCVAE, None)
# _, _ = result.pop(our0, None), result_time.pop(our0, None)
# _, _ = result.pop(our_GRE, None), result_time.pop(our_GRE, None)
# _, _ = result.pop(tupl, None), result_time.pop(tupl, None)
# for k in [f"{our_tupl}_{p}" for p in GENERATION_POLICIES]:
#     _, _ = result.pop(k, None), result_time.pop(k, None)


## Base

In [5]:
def main_base(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    classifier = get_trained_classifier_Base(
        data_loaders=data_loaders,
        NUM_LABELS=NUM_LABELS,
        device=device)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [6]:
if base not in result or base not in result_time:
    start = time.time()
    result[base] = run_all_senario(main_base, DOMAIN_SET)
    result_time[base] = time.time() - start
    save_results()


# GZSDA

In [7]:
def main_gzsda(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [8]:
if CCVAE not in result or CCVAE not in result_time:
    start = time.time()
    result[CCVAE] = run_all_senario(main_gzsda, DOMAIN_SET)
    result_time[CCVAE] = time.time() - start
    save_results()


## m0

In [9]:
def main_m0(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [10]:
if our0 not in result or our0 not in result_time:
    start = time.time()
    result[our0] = run_all_senario(main_m0, DOMAIN_SET)
    result_time[our0] = time.time() - start
    save_results()


## m1: seperate after encoder

In [11]:
def main_m1(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE_with_domain_classifier(
        data_loaders=data_loaders,
        args=args,
        device=device)
        
    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [12]:
if our_GRE not in result or our_GRE not in result_time:
    start = time.time()
    result[our_GRE] = run_all_senario(main_m1, DOMAIN_SET)
    result_time[our_GRE] = time.time() - start
    save_results()


## TUPL

In [13]:
def main_tupl(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    acc_s, acc_u, h = run_tupl(
        data_root="./data/",
        dataset="officehome",
        source=args.sourceDomainIndex,
        target=args.targetDomainIndex,
        trial=args.trialIndex,
        seed=args.seed,
        device=device,
        quiet=True,
        return_model=False,
    )
    print('seen acc:{:2.4f}, unseen acc:{:2.4f}, H:{:2.4f}'.format(acc_s / 100, acc_u / 100, h / 100))
    return None, None, acc_s / 100.0, acc_u / 100.0

In [14]:
if tupl not in result or tupl not in result_time:
    start = time.time()
    result[tupl] = run_all_senario(main_tupl, DOMAIN_SET)
    result_time[tupl] = time.time() - start
    save_results()


## our_TUPL: m1 VAE + TUPL

In [15]:
def main_m1_tupl(args, policy="real_plus_src2tgt"):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    acc_s, acc_u, h = run_m1_tupl(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS,
        policy=policy,
        device=device,
        quiet=True,
        seed=args.seed,
    )
    print('seen acc:{:2.4f}, unseen acc:{:2.4f}, H:{:2.4f}'.format(acc_s / 100, acc_u / 100, h / 100))
    return None, None, acc_s / 100.0, acc_u / 100.0

In [16]:
for p in GENERATION_POLICIES:
    key = f"{our_tupl}_{p}"
    if key not in result or key not in result_time:
        start = time.time()
        result.update(run_all_senario_m1_tupl(
            DOMAIN_SET=DOMAIN_SET,
            DATA_DIR=DATA_DIR,
            DATASET_DETAILS=DATASET_DETAILS,
            policies=[p],
        ))
        result_time[key] = time.time() - start
        save_results()


## Merge results

In [17]:
save_results()


In [18]:
# ignore our0
_, _ = result.pop(our0, None), result_time.pop(our0, None)
_, _ = result.pop("our_TUPL_real_plus_src2tgt", None), result_time.pop("our_TUPL_real_plus_src2tgt", None)
_, _ = result.pop("our_TUPL_real_plus_src2tgt_unseen", None), result_time.pop("our_TUPL_real_plus_src2tgt_unseen", None)
# _, _ = result.pop("our_TUPL_interp_src2tgt", None), result_time.pop("our_TUPL_interp_src2tgt", None)


In [19]:
import pandas as pd

n_senario = len(DOMAIN_SET) * (len(DOMAIN_SET) - 1)
n_trial = 5


df_time = pd.DataFrame(
    [
        {
            "method": method,
            "n_senario": n_senario,
            "n_trial": n_trial,
            "total_trial": n_senario * n_trial,
            "total_time": round(total_time, 1),
        }
        for method, total_time in result_time.items()
    ]
)
df_time.to_csv(RESULT_TIME_CSV_PATH, index=False)
# df_time


In [20]:
import pandas as pd
import re

rows = [(k, m, result[m][k]) for m in result for k in result[m]]
df = pd.DataFrame(rows, columns=['domain', 'method', 'values'])

def extract_metrics(text):
    matches = dict(re.findall(r'(\w+):\s+([\d.]+\s*±\s*[\d.]+)', text))
    return pd.Series(matches)

df[['seen', 'unseen', 'H-mean']] = df['values'].apply(extract_metrics)
df = df[['domain', 'method', 'seen', 'unseen', 'H-mean']]
df['method'] = pd.Categorical(
    df['method'],
    categories=[base, CCVAE, tupl, our0, our_GRE] + [f"{our_tupl}_{p}" for p in GENERATION_POLICIES],
    ordered=True,
)
df = df.sort_values(['domain', 'method']).reset_index(drop=True)

df

,domain,method,seen,unseen,H-mean
0,Art -> Clipart,base,71.69 ± 0.55,28.16 ± 0.58,40.41 ± 0.52
1,Art -> Clipart,CCVAE,66.56 ± 0.56,41.56 ± 0.79,51.13 ± 0.48
2,Art -> Clipart,TUPL,59.92 ± 1.05,39.21 ± 1.15,47.36 ± 0.96
3,Art -> Clipart,our_GRE,63.98 ± 0.60,45.21 ± 0.76,52.96 ± 0.50
4,Art -> Clipart,our_TUPL_interp_src2tgt,55.60 ± 0.77,32.40 ± 1.43,40.87 ± 1.18
5,Art -> Product,base,89.66 ± 0.14,53.13 ± 1.24,66.68 ± 0.96
6,Art -> Product,CCVAE,87.17 ± 0.38,65.26 ± 0.98,74.62 ± 0.60
7,Art -> Product,TUPL,78.04 ± 0.85,58.08 ± 1.20,66.55 ± 0.72
8,Art -> Product,our_GRE,85.41 ± 0.48,68.76 ± 1.19,76.15 ± 0.67
9,Art -> Product,our_TUPL_interp_src2tgt,73.23 ± 0.92,49.84 ± 0.50,59.30 ± 0.38


In [21]:
df.to_csv(RESULT_CSV_PATH, index=False)